In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import wbgapi as wb

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"SSL_VERIFY: {SSL_VERIFY}")

Log loaded. Rows: 8
PROJECT_ROOT: /Users/boulanger/Documents/governance-framework
SSL_VERIFY: True


## WDI Pipeline

**Source:** World Bank World Development Indicators
**Access:** Automated via `wbgapi` — no manual step required
**Download instructions:** See `docs/instructions_data_maintenance.md` — WB_WDI section

### Framework usage
| Indicator | Dimension | Concept |
|-----------|-----------|---------|
| Primary completion rate | Education | Service delivery |
| Primary enrollment (gross) | Education | Service delivery |
| Secondary enrollment (gross) | Education | Service delivery |
| Under-5 mortality rate | Health | Service delivery |
| DPT immunization | Health | Service delivery |
| Measles immunization | Health | Service delivery |
| Maternal mortality ratio | Health | Service delivery |
| Access to electricity | Infrastructure | Service delivery |
| Basic drinking water access | Infrastructure | Service delivery |
| Basic sanitation access | Infrastructure | Service delivery |

In [21]:
import wbgapi as wb
import pandas as pd
from datetime import datetime
import signal

# WDI indicators for service delivery
WDI_INDICATORS = {
    # Education
    'SE.PRM.CMPT.ZS': 'wdi_primary_completion_rate',
    'SE.PRM.ENRR':    'wdi_primary_enrollment_gross',
    'SE.SEC.ENRR':    'wdi_secondary_enrollment_gross',
    # Health outcomes
    'SH.DYN.MORT':    'wdi_mortality_under5',
    'SH.IMM.IDPT':    'wdi_immunization_dpt',
    'SH.IMM.MEAS':    'wdi_immunization_measles',
    'SH.STA.MMRT':    'wdi_maternal_mortality',
    # Health system capacity — sourced from WHO, distributed via WDI
    'SH.MED.PHYS.ZS': 'wdi_physicians_per_1000',
    'SH.MED.NUMW.P3': 'wdi_nurses_per_1000',
    'SH.MED.BEDS.ZS': 'wdi_hospital_beds_per_1000',
    'SH_UHC_SCI':     'wdi_uhc_coverage_index',
    # Infrastructure
    'EG.ELC.ACCS.ZS': 'wdi_electricity_access',
    'SH.H2O.BASW.ZS': 'wdi_basic_water_access',
    'SH.STA.BASS.ZS': 'wdi_basic_sanitation_access',
    # Education system quality — sourced from UNESCO UIS, distributed via WDI
    'SE.XPD.TOTL.GD.ZS': 'wdi_education_expenditure_gdp',
    'SE.XPD.TOTL.GB.ZS': 'wdi_education_expenditure_govt',
    'SE.PRM.ENRL.TC.ZS': 'wdi_pupil_teacher_ratio_primary',
    'SE.SEC.ENRL.TC.ZS': 'wdi_pupil_teacher_ratio_secondary',
    # Gender equality — WBL 2.0 indicators, distributed via WDI
    'GD_WBL_OVL_LAW': 'wbl_legal_framework',
    'GD_WBL_OVL_SFR': 'wbl_supportive_framework',
    'GD_WBL_OVL_ENF': 'wbl_enforcement_perceptions',
    # Trade administration — World Bank Logistics Performance Index
    'LP.LPI.OVRL.XQ': 'wdi_lpi_overall',
    # Human capital — HCI+ overall (standard HCI not available via API)
    'HD_HCIP_OVRL_TO': 'wdi_hci_plus_overall',
    # IP protection — sourced from WIPO, distributed via WDI
    'IP.PAT.RESD': 'wdi_patent_applications_resident',
    'IP.PAT.NRES': 'wdi_patent_applications_nonresident',
    'IP.TMK.RSCT': 'wdi_trademark_applications_resident',
    'IP.TMK.NRCT': 'wdi_trademark_applications_nonresident',
    # Social protection coverage — ILO data distributed via World Bank
    'per_allsp.cov_pop_tot':   'wdi_social_protection_coverage',
    'per_sa_allsa.cov_pop_tot': 'wdi_safety_net_coverage',
    'per_si_allsi.cov_pop_tot': 'wdi_social_insurance_coverage',
    
}

# Apply SSL setting
if not SSL_VERIFY:
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    wb.fetch_options = {'verify': False}

# Set WDI database
wb.db = 2

def fetch_with_retry(code, name, max_attempts=3):
    """Fetch a single WDI indicator with retry logic."""
    for attempt in range(1, max_attempts + 1):
        try:
            df = wb.data.DataFrame(
                code,
                time=range(FRAMEWORK_START_YEAR, CURRENT_YEAR + 1),
                labels=True
            )
            df = df.reset_index()
            df['indicator'] = name
            return df
        except Exception as e:
            print(f"  Attempt {attempt} failed: {e}")
            if attempt == max_attempts:
                print(f"  All attempts failed for {code}")
                return None

# Fetch one indicator at a time with retry
frames = []
failed = []
for code, name in WDI_INDICATORS.items():
    print(f"Fetching {code}...")
    df = fetch_with_retry(code, name)
    if df is not None:
        frames.append(df)
        print(f"  OK: {df.shape}")
    else:
        failed.append(code)

print(f"\nDownloaded {len(frames)} indicators")
if failed:
    print(f"Failed: {failed}")

Fetching SE.PRM.CMPT.ZS...
  OK: (266, 39)
Fetching SE.PRM.ENRR...
  OK: (266, 39)
Fetching SE.SEC.ENRR...
  OK: (266, 39)
Fetching SH.DYN.MORT...
  OK: (266, 39)
Fetching SH.IMM.IDPT...
  OK: (266, 39)
Fetching SH.IMM.MEAS...
  OK: (266, 39)
Fetching SH.STA.MMRT...
  OK: (266, 39)
Fetching SH.MED.PHYS.ZS...
  OK: (266, 39)
Fetching SH.MED.NUMW.P3...
  OK: (266, 39)
Fetching SH.MED.BEDS.ZS...
  OK: (266, 39)
Fetching SH_UHC_SCI...
  OK: (266, 39)
Fetching EG.ELC.ACCS.ZS...
  OK: (266, 39)
Fetching SH.H2O.BASW.ZS...
  OK: (266, 39)
Fetching SH.STA.BASS.ZS...
  OK: (266, 39)
Fetching SE.XPD.TOTL.GD.ZS...
  OK: (266, 39)
Fetching SE.XPD.TOTL.GB.ZS...
  OK: (266, 39)
Fetching SE.PRM.ENRL.TC.ZS...
  OK: (266, 39)
Fetching SE.SEC.ENRL.TC.ZS...
  OK: (266, 39)
Fetching GD_WBL_OVL_LAW...
  OK: (266, 39)
Fetching GD_WBL_OVL_SFR...
  OK: (266, 39)
Fetching GD_WBL_OVL_ENF...
  OK: (266, 39)
Fetching LP.LPI.OVRL.XQ...
  OK: (266, 39)
Fetching HD_HCIP_OVRL_TO...
  OK: (266, 39)
Fetching IP.PAT.RESD

In [22]:
# Melt each frame from wide to long, then combine
long_frames = []
for df in frames:
    year_cols = [c for c in df.columns if c.startswith('YR')]
    melted = df.melt(
        id_vars=['economy', 'Country', 'indicator'],
        value_vars=year_cols,
        var_name='year_str',
        value_name='value'
    )
    melted['year'] = melted['year_str'].str.replace('YR', '').astype(int)
    melted = melted.drop(columns=['year_str'])
    long_frames.append(melted)

wdi_long = pd.concat(long_frames, ignore_index=True)

# Pivot to wide format — one column per indicator
wdi_wide = wdi_long.pivot_table(
    index=['economy', 'Country', 'year'],
    columns='indicator',
    values='value'
).reset_index()

wdi_wide.columns.name = None
wdi_wide = wdi_wide.rename(columns={
    'economy': 'country_code',
    'Country': 'country_name'
})

# Sort
wdi_wide = wdi_wide.sort_values(['country_code', 'year']).reset_index(drop=True)

print(f"Shape: {wdi_wide.shape}")
print(f"Years: {wdi_wide['year'].min()} — {wdi_wide['year'].max()}")
print(f"Countries: {wdi_wide['country_code'].nunique()}")
print(f"\nMissing values (%):")
missing_pct = (wdi_wide.isnull().sum() / len(wdi_wide) * 100).round(1)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

Shape: (9478, 33)
Years: 1990 — 2025
Countries: 265

Missing values (%):
wbl_enforcement_perceptions               98.0
wbl_supportive_framework                  97.9
wbl_legal_framework                       97.9
wdi_social_insurance_coverage             94.5
wdi_safety_net_coverage                   94.0
wdi_hci_plus_overall                      93.9
wdi_social_protection_coverage            93.6
wdi_lpi_overall                           85.2
wdi_trademark_applications_resident       77.8
wdi_trademark_applications_nonresident    77.6
wdi_patent_applications_resident          62.7
wdi_nurses_per_1000                       62.6
wdi_patent_applications_nonresident       60.5
wdi_pupil_teacher_ratio_secondary         57.8
wdi_physicians_per_1000                   57.3
wdi_hospital_beds_per_1000                50.9
wdi_pupil_teacher_ratio_primary           46.8
wdi_education_expenditure_gdp             45.7
wdi_education_expenditure_govt            45.7
wdi_primary_completion_rate       

In [23]:
# Get metadata from API automatically
wb.db = 2
wdi_source_meta = next(s for s in wb.source.list() if s['id'] == '2')
data_as_of_date = wdi_source_meta['lastupdated'][:7]
latest_year = str(int(wdi_wide['year'].max()))

print(f"Data as of: {data_as_of_date}")
print(f"Latest year: {latest_year}")

# Save to processed
output_path = os.path.join(PROCESSED_DIR, "wdi_clean.csv")
wdi_wide.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {wdi_wide.shape}")

# Update download log
update_entry(
    "WB_WDI",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=data_as_of_date,
    local_filename="wdi_clean.csv",
    latest_available_version=latest_year,
    notes="30 service delivery indicators. Fetched one at a time to avoid API timeout. 266 economies including aggregates."
)

print_entry("WB_WDI")

Data as of: 2026-04
Latest year: 2025
Written: /Users/boulanger/Documents/governance-framework/data/processed/wdi_clean.csv
Shape: (9478, 33)
[download_log] Updated entry for WB_WDI
  source_id: WB_WDI
  last_attempted_date: 2026-05-29
  last_successful_download_date: 2026-06-01
  data_as_of_date: 2026-04
  local_filename: wdi_clean.csv
  latest_available_version: 2025
  no_update_reason: nan
  notes: 30 service delivery indicators. Fetched one at a time to avoid API timeout. 266 economies including aggregates.


In [20]:
wb.db = 2
for code in ['per_allsp.cov_pop_tot', 'per_sa_allsa.cov_pop_tot', 'per_si_allsi.cov_pop_tot']:
    try:
        df = wb.data.DataFrame(code, time=range(2020, 2023), labels=False)
        print(f"IN WDI: {code} — shape {df.shape}")
    except Exception as e:
        print(f"NOT via wbgapi: {code}")

IN WDI: per_allsp.cov_pop_tot — shape (266, 3)
IN WDI: per_sa_allsa.cov_pop_tot — shape (266, 3)
IN WDI: per_si_allsi.cov_pop_tot — shape (266, 3)
